# AutoClinic Consult — Gemma RAG over a Medical Q&A Dataset

This notebook is the research/prototyping counterpart of `autoclinic-consult-api`'s
`lib/dataset.js`, `lib/vectorstore.js`, and `lib/gemma.js`. It:

1. Pulls a small slice of the **MedMCQA** dataset (`openlifescienceai/medmcqa`)
   from Hugging Face, filtered on the same four parameters the Node.js
   dashboard exposes: **subject, topic, choice type, correct option**.
2. Chunks and embeds the records (`sentence-transformers`, a small local
   embedding model — no ChromaDB, matching the Node backend's in-memory
   vector-store design).
3. Retrieves top-k relevant chunks for a question via cosine similarity.
4. Calls a **Gemma** model (`google/gemma-2-2b-it` via 🤗 Transformers, or the
   Google AI Studio API if you set `GOOGLE_GENERATIVE_AI_API_KEY`) to answer
   **only** from the retrieved context — the same "cite-or-refuse" RAG
   pattern used server-side.
5. Prints results in the exact JSON shape (`answer`, `provider`, `model`,
   `sources[]`) that `autoclinic-consult-api`'s `POST /api/query` returns, so
   this notebook's output can be dropped straight into the Node app or used
   to sanity-check it after deployment.

> **Note on execution environment:** running Gemma locally needs a GPU (or a
> lot of patience on CPU) and network access to Hugging Face / Google AI
> Studio. If you're running this in a sandboxed/offline environment, use the
> `GEMMA_PROVIDER = "mock"` toggle in the config cell to exercise the full
> pipeline with a stub generator, then flip it to `"transformers"` or
> `"google"` once you have GPU/network access.


## 1. Install dependencies

(Run once — commented out for repeated re-runs.)

In [1]:
# !pip install -q datasets sentence-transformers transformers accelerate torch google-generativeai


## 2. Configuration — mirrors `.env.example` in the Node backend

In [2]:
import os

HF_DATASET_ID = os.environ.get("HF_DATASET_ID", "openlifescienceai/medmcqa")
SPLIT = "train"

# Four dashboard-style selection parameters (leave as None for "Any")
FILTER_SUBJECT_NAME = "Medicine"        # e.g. "Medicine", "Pharmacology", "Anatomy"
FILTER_TOPIC_NAME = None                # substring match, e.g. "Cardiovascular"
FILTER_CHOICE_TYPE = "single"           # "single" | "multi"
FILTER_CORRECT_OPTION = None            # "a" | "b" | "c" | "d"

RECORD_LIMIT = 150          # keep this small — "small medical dataset" slice
TOP_K = 4

# "mock" (offline, no GPU/network needed) | "transformers" (local Gemma) | "google" (AI Studio API)
GEMMA_PROVIDER = "mock"
GEMMA_MODEL_TRANSFORMERS = "google/gemma-2-2b-it"
GEMMA_MODEL_GOOGLE = "gemma-3-27b-it"
GOOGLE_GENERATIVE_AI_API_KEY = os.environ.get("GOOGLE_GENERATIVE_AI_API_KEY", "")

print("Config loaded.", HF_DATASET_ID, FILTER_SUBJECT_NAME, FILTER_CHOICE_TYPE, GEMMA_PROVIDER)


Config loaded. openlifescienceai/medmcqa Medicine single mock


## 3. Load and filter MedMCQA (four selection parameters)

In [3]:
from datasets import load_dataset

COP_LETTERS = ["a", "b", "c", "d"]

def load_records(limit=RECORD_LIMIT):
    ds = load_dataset(HF_DATASET_ID, split=SPLIT, streaming=True)
    matched = []
    for row in ds:
        cop_letter = COP_LETTERS[row["cop"]] if isinstance(row.get("cop"), int) else row.get("cop")

        if FILTER_SUBJECT_NAME and (row.get("subject_name") or "").lower() != FILTER_SUBJECT_NAME.lower():
            continue
        if FILTER_TOPIC_NAME and FILTER_TOPIC_NAME.lower() not in (row.get("topic_name") or "").lower():
            continue
        if FILTER_CHOICE_TYPE and (row.get("choice_type") or "").lower() != FILTER_CHOICE_TYPE.lower():
            continue
        if FILTER_CORRECT_OPTION and cop_letter != FILTER_CORRECT_OPTION.lower():
            continue

        matched.append({
            "id": row.get("id"),
            "question": row.get("question"),
            "options": {"a": row.get("opa"), "b": row.get("opb"), "c": row.get("opc"), "d": row.get("opd")},
            "correct_option": cop_letter,
            "choice_type": row.get("choice_type"),
            "explanation": row.get("exp"),
            "subject_name": row.get("subject_name"),
            "topic_name": row.get("topic_name"),
        })
        if len(matched) >= limit:
            break
    return matched

records = load_records()
print(f"Matched {len(records)} MedMCQA records for the selected filters.")
records[:2]


README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

Matched 150 MedMCQA records for the selected filters.


[{'id': 'f5df7424-6485-43fa-ba98-6de498561a76',
  'question': 'The most common cause of renal scaring in a 3 year old child is -',
  'options': {'a': 'Trauma',
   'b': 'Tuberculosis',
   'c': 'Vesicoureteral reflux induced pyelonephritis',
   'd': 'Interstitial nephritis'},
  'correct_option': 'c',
  'choice_type': 'single',
  'explanation': 'Chronic pyelonephritis is characterized by renal inflammation and scarring induced by recurrent or persistent renal infection, vesicoureteral reflux, or other causes of urinary tract obstruction. VUR is a congenital condition that results from incompetence of the ureterovesical valve due to a sho intramural segment Ref Harrison20th edition pg 234',
  'subject_name': 'Medicine',
  'topic_name': 'Kidney'},
 {'id': '9595ba1f-bc34-42ab-8603-45961b925ad0',
  'question': 'Of the various modalities used in the treatment of re-threatening effects of hyperkalemia which one of the following as the most rapid onset of action ?',
  'options': {'a': 'Hemodialy

## 4. Render records to text (same shape as `lib/dataset.js`'s `recordToText`)

In [4]:
def record_to_text(r):
    opts = r["options"]
    lines = [
        f"Question: {r['question']}",
        f"Options: (a) {opts['a']}  (b) {opts['b']}  (c) {opts['c']}  (d) {opts['d']}",
        f"Correct answer: {r['correct_option']}",
    ]
    if r.get("explanation"):
        lines.append(f"Explanation: {r['explanation']}")
    lines.append(f"Subject: {r['subject_name']} | Topic: {r['topic_name']} | Choice type: {r['choice_type']}")
    return "\n".join(lines)

corpus_texts = [record_to_text(r) for r in records]
print(corpus_texts[0] if corpus_texts else "(no records matched — widen your filters)")


Question: The most common cause of renal scaring in a 3 year old child is -
Options: (a) Trauma  (b) Tuberculosis  (c) Vesicoureteral reflux induced pyelonephritis  (d) Interstitial nephritis
Correct answer: c
Explanation: Chronic pyelonephritis is characterized by renal inflammation and scarring induced by recurrent or persistent renal infection, vesicoureteral reflux, or other causes of urinary tract obstruction. VUR is a congenital condition that results from incompetence of the ureterovesical valve due to a sho intramural segment Ref Harrison20th edition pg 234
Subject: Medicine | Topic: Kidney | Choice type: single


## 5. Chunk + embed (in-memory, no ChromaDB)

Uses a small local `sentence-transformers` model for embeddings, matching the
Node backend's choice to keep everything in one process with no external
vector database — just a Python list of `(text, vector, metadata)` tuples,
searched with cosine similarity.

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def chunk_text(text, chunk_size=400, overlap=40):
    # Simple sentence-aware chunker, analogous to LlamaIndex's SentenceSplitter
    # used server-side in lib/vectorstore.js.
    words = text.split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

CHUNKS = []  # list of dicts: text, vector, metadata
for r, text in zip(records, corpus_texts):
    for piece in chunk_text(text):
        CHUNKS.append({
            "text": piece,
            "metadata": {
                "subject_name": r["subject_name"],
                "topic_name": r["topic_name"],
                "choice_type": r["choice_type"],
                "correct_option": r["correct_option"],
            },
        })

if CHUNKS:
    vectors = embedder.encode([c["text"] for c in CHUNKS], normalize_embeddings=True)
    for c, v in zip(CHUNKS, vectors):
        c["vector"] = v

print(f"Indexed {len(CHUNKS)} chunks in memory.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Indexed 156 chunks in memory.


## 6. Retrieval — cosine similarity top-k

In [6]:
def retrieve(question, top_k=TOP_K):
    if not CHUNKS:
        return []
    q_vec = embedder.encode([question], normalize_embeddings=True)[0]
    scored = []
    for c in CHUNKS:
        score = float(np.dot(q_vec, c["vector"]))
        scored.append({**c, "score": score})
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


## 7. Gemma generation — RAG answer with citations

Three interchangeable backends, selected by `GEMMA_PROVIDER`:

- **`mock`** — a deterministic stub so the whole pipeline runs offline with
  no GPU/API key, useful for CI or a sandboxed environment like this one.
- **`transformers`** — loads `google/gemma-2-2b-it` locally via 🤗
  Transformers (needs a GPU for reasonable latency, and a Hugging Face token
  with Gemma's license accepted).
- **`google`** — calls the same Google AI Studio Generative Language API the
  Node backend uses (`GEMMA_MODEL_GOOGLE`, e.g. `gemma-3-27b-it`).

In [7]:
def build_prompt(question, chunks):
    numbered = "\n\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(chunks))
    system = (
        "You are a careful clinical-reference assistant. Answer the user's question using ONLY "
        "the numbered context below. Cite snippet numbers you used, e.g. [1][3]. If the context "
        "does not contain the answer, say so plainly instead of guessing. This is for study/"
        "reference purposes, not a diagnosis of any real patient."
    )
    return f"{system}\n\nContext:\n{numbered}\n\nQuestion: {question}\n\nAnswer:"


def gemma_generate_mock(question, chunks):
    if not chunks:
        return "No context was retrieved — ingest a broader set of records first.", "mock", "mock-gemma"
    top = chunks[0]
    answer = (
        f"Based on snippet [1] (subject: {top['metadata']['subject_name']}, "
        f"topic: {top['metadata']['topic_name']}), the correct option is "
        f"'{top['metadata']['correct_option']}'. [1]"
    )
    return answer, "mock", "mock-gemma"


def gemma_generate_transformers(question, chunks):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch
    tok = AutoTokenizer.from_pretrained(GEMMA_MODEL_TRANSFORMERS)
    model = AutoModelForCausalLM.from_pretrained(
        GEMMA_MODEL_TRANSFORMERS, torch_dtype=torch.bfloat16, device_map="auto"
    )
    prompt = build_prompt(question, chunks)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=256)
    text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return text, "transformers-local", GEMMA_MODEL_TRANSFORMERS


def gemma_generate_google(question, chunks):
    import google.generativeai as genai
    genai.configure(api_key=GOOGLE_GENERATIVE_AI_API_KEY)
    model = genai.GenerativeModel(GEMMA_MODEL_GOOGLE)
    prompt = build_prompt(question, chunks)
    resp = model.generate_content(prompt)
    return resp.text, "google-ai-studio", GEMMA_MODEL_GOOGLE


def rag_answer(question, top_k=TOP_K):
    chunks = retrieve(question, top_k)
    if GEMMA_PROVIDER == "transformers":
        text, provider, model = gemma_generate_transformers(question, chunks)
    elif GEMMA_PROVIDER == "google":
        text, provider, model = gemma_generate_google(question, chunks)
    else:
        text, provider, model = gemma_generate_mock(question, chunks)

    return {
        "answer": text,
        "provider": provider,
        "model": model,
        "sources": [
            {
                "text": c["text"],
                "score": c["score"],
                "subjectName": c["metadata"]["subject_name"],
                "topicName": c["metadata"]["topic_name"],
                "choiceType": c["metadata"]["choice_type"],
                "correctOption": c["metadata"]["correct_option"],
            }
            for c in chunks
        ],
    }


## 8. Run it

In [8]:
import json

result = rag_answer("What is the correct management option discussed in the ingested Medicine questions?")
print(json.dumps(result, indent=2)[:2000])


{
  "answer": "Based on snippet [1] (subject: Medicine, topic: Bacteriology), the correct option is 'a'. [1]",
  "provider": "mock",
  "model": "mock-gemma",
  "sources": [
    {
      "text": "Question: DOC for listeria meningitis: Options: (a) Ampicillin (b) Cefotaxime (c) Cefotriaxone (d) Ciprofloxacin Correct answer: a Explanation: Ans. is 'a' i.e., Ampicillin Treatment of listeria infectiono The antibiotic of choice for listeria infection is ampicillin or penicillin G.Antibiotic regimens for listeria infection||||First line regimensPenicillin allergic patientsAlternative drugso Ampicillin or Penicillin is the drug of choiceo Trimethoprim sulphame- thoxazoleo Imipenem and meropenemo Other antibiotic that are less effective# Vancomycin# Erythromycin# Chloramphenicol Subject: Medicine | Topic: Bacteriology | Choice type: single",
      "score": 0.4878019690513611,
      "subjectName": "Medicine",
      "topicName": "Bacteriology",
      "choiceType": "single",
      "correctOption": 

## 9. Bridging notebook results into `autoclinic-consult-api`

This notebook's `rag_answer()` output is intentionally identical in shape to
`POST /api/query`'s JSON response in the Node backend
(`answer`, `provider`, `model`, `sources[]` with the same field names). That
means:

- The **retrieval + chunking logic prototyped here** (Sections 5–6) is what
  `lib/vectorstore.js` implements server-side with `llamaindex`'s
  `SentenceSplitter` + cosine similarity over an in-memory array.
- The **filtering logic prototyped here** (Section 3) is what
  `lib/dataset.js` implements server-side against the Hugging Face
  `datasets-server` REST API.
- The **RAG prompt + citation pattern prototyped here** (Section 7) is what
  `lib/gemma.js` implements server-side via the Vercel AI SDK.

To go from notebook to production API: deploy `autoclinic-consult-api/` to
Render (see its `render.yaml` / README), set `GEMMA_PROVIDER=google` and
`GOOGLE_GENERATIVE_AI_API_KEY` in the Render dashboard, call
`POST /api/ingest` with the same four filters used in Section 2 above, then
`POST /api/query` — and point the SwiftUI app's `APIClient.baseURL` at the
deployed Render URL.